In [ ]:
!pip install ortools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 10.0 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


# Pregunta 1

Un hospital debe asignar a 3 médicos (M1,M2,M3) a 3 turnos diferentes (Mañana, Tarde, Noche).

Cada médico debe tener exactamente un turno, y ningún turno puede estar sin médico. Debido a restricciones médicas y de agenda:  

M1 no puede trabajar en la Noche.  
M2 no puede trabajar en la Mañana.  
M3 debe trabajar después de M1 pero antes de M2 (en un turno cronológicamente posterior).  
Modele formalmente este problema como un Problema de Satisfacción de Restricciones (CSP). Defina claramente:  
(1 punto) El conjunto de Variables (X).  
(1 punto) El dominio de cada variable (D), indicando cómo mapeará los turnos a valores numéricos para representar el tiempo.  
(1 punto) El conjunto de restricciones formales (C).  
(1 punto) ¿La solución es única? o ¿existe más de una?.  
(1 punto) Imprima el tiempo de ejecución del modelo con la función walltime del solver.  


In [ ]:
from ortools.sat.python import cp_model

# 1. Modelo
model = cp_model.CpModel()

# Variables y Dominios
m1 = model.NewIntVar(1, 3, 'M1')
m2 = model.NewIntVar(1, 3, 'M2')
m3 = model.NewIntVar(1, 3, 'M3')

# Restricciones
model.AddAllDifferent([m1, m2, m3])
model.Add(m1 != 3)
model.Add(m2 != 1)
model.Add(m1 < m3)
model.Add(m3 < m2)

# Solver
solver = cp_model.CpSolver()
status = solver.Solve(model)

if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    print(f"Solución: M1={solver.Value(m1)} (Mañana), M2={solver.Value(m2)} (Noche), M3={solver.Value(m3)} (Tarde)")
    print(f"Tiempo de ejecución: {solver.WallTime()} segundos")

Solución: M1=1 (Mañana), M2=3 (Noche), M3=2 (Tarde)
Tiempo de ejecución: 0.005050285000000001 segundos


# Pregunta 2

In [ ]:
# trivial

# Pregunta 3

Usted dirige una obra con un equipo máximo de 5 obreros disponibles en todo momento (Capacidad). Tiene 3 tareas con los siguientes requerimientos:

Tarea 1: Duración = 2 días, Obreros necesarios = 3  
Tarea 2: Duración = 3 días, Obreros necesarios = 2  
Tarea 3: Duración = 1 día, Obreros necesarios = 4  
Todas las tareas deben completarse en un horizonte temporal de 10 días (del día 0 al 10).  
(1 punto) Explique cómo funciona internamente la restricción global cumulative.  
(1 punto) ¿Qué ventaja tiene evaluar el "perfil de recursos" globalmente en el tiempo en comparación a intentar modelar esto de forma ingenua verificando que las tareas no se superpongan en el tiempo por pares?  
Asumiendo que las duraciones y demandas son constantes, escriba el código en OR-Tools para:  
(1 punto) Declarar las variables de inicio (start), fin (end) e intervalo (interval) para cada una de las 3 tareas.  
(2 puntos) Agregar la restricción global AddCumulative para asegurar que nunca se exceda el límite de los 5 obreros.  

In [ ]:
from ortools.sat.python import cp_model

# Inicializar modelo
model = cp_model.CpModel()

# Parámetros del problema
horizon = 10     # Días del 0 al 10 (10 días de capacidad, indexados 0 a 9)
capacity = 5     # Límite máximo de obreros

durations = [2, 3, 1]  # Duraciones de Tarea 1, 2 y 3
demands = [3, 2, 4]    # Obreros necesarios para Tarea 1, 2 y 3
num_tasks = 3

# 1. Variables: start_x[t][i] es True (1) si la Tarea i inicia EXACTAMENTE en el día t
start_x = {}
for i in range(num_tasks):
    for t in range(horizon):
        start_x[t, i] = model.NewBoolVar(f'start_tarea_{i}_dia_{t}')
# Se crean las variables start_tarea_0_dia_0 ... 

# 2. Restricciones de inicio y fin dentro del horizonte
for i in range(num_tasks):
    valid_starts = []
    for t in range(horizon):
        # Una tarea solo puede iniciar si tiene días suficientes para terminar antes del horizonte
        if t + durations[i] <= horizon:
            valid_starts.append(start_x[t, i])
        else:
            # Si inicia aquí, sobrepasaría el horizonte, así que lo forzamos a 0
            model.Add(start_x[t, i] == 0)

    # Cada tarea debe iniciar exactamente una vez.
    model.AddExactlyOne(valid_starts)
    # El solver elegirá exactamente un día de inicio cuando se resuelva el modelo con CpSolver().

# 3. Restricción manual de recursos (Reemplazo de AddCumulative)
# Para cada día 't', la suma de obreros de las tareas activas no debe superar la capacidad (5)
for t in range(horizon):
    obreros_activos_dia_t = []

    for i in range(num_tasks):
        tareas_empezadas = []
        # La tarea 'i' está activa en el día 't' si inició en algún día 't_prime' tal que:
        # t_prime <= t (ya empezó)  Y  t_prime > t - duracion (aún no termina)
        for t_prime in range(max(0, t - durations[i] + 1), t + 1):
            tareas_empezadas.append(start_x[t_prime, i])

        # Si la lista tiene booleanos, multiplicamos su suma (que será 0 o 1) por la demanda
        if tareas_empezadas:
            obreros_activos_dia_t.append(sum(tareas_empezadas) * demands[i])

    # Agregamos la restricción de capacidad al modelo para ese día específico
    if obreros_activos_dia_t:
        model.Add(sum(obreros_activos_dia_t) <= capacity)

# --- Variables auxiliares enteras para poder imprimir fácilmente los resultados ---
start_vars = []
end_vars = []
for i in range(num_tasks):
    s = model.NewIntVar(0, horizon, f'start_t{i+1}')
    e = model.NewIntVar(0, horizon, f'end_t{i+1}')
    # El día de inicio real es la suma de (t * booleano_de_inicio_en_t)
    model.Add(s == sum(t * start_x[t, i] for t in range(horizon)))
    model.Add(e == s + durations[i])
    start_vars.append(s)
    end_vars.append(e)

# 4. Solucionador (Solver) e impresión de resultados
solver = cp_model.CpSolver()
status = solver.Solve(model)

if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    print("¡Solución encontrada!\n")
    for i in range(num_tasks):
        inicio = solver.Value(start_vars[i])
        fin = solver.Value(end_vars[i])
        print(f"Tarea {i+1} (Requiere {demands[i]} obreros): Inicia en día {inicio}, Termina en día {fin}")

    print(f"\nTiempo de ejecución: {solver.WallTime():.4f} segundos")
else:
    print("No se encontró solución factible que cumpla con el límite de obreros.")

¡Solución encontrada!

Tarea 1 (Requiere 3 obreros): Inicia en día 8, Termina en día 10
Tarea 2 (Requiere 2 obreros): Inicia en día 5, Termina en día 8
Tarea 3 (Requiere 4 obreros): Inicia en día 0, Termina en día 1

Tiempo de ejecución: 0.0088 segundos



## 3. Que no falten obreros

Hay tres tareas:

- Tarea 1 dura 2 días y usa 3 obreros.
- Tarea 2 dura 3 días y usa 2 obreros.
- Tarea 3 dura 1 día y usa 4 obreros.

El código revisa cada día y pregunta: “¿Qué tareas están trabajando hoy y cuántos obreros necesitan juntas?”

Por ejemplo, si en un día trabajan la tarea 1 y la tarea 2:

3 obreros + 2 obreros = 5

Eso está permitido, porque el máximo es 5.

Pero si trabajan juntas la tarea 1 y la tarea 3:

3 obreros + 4 obreros = 7

Eso no está permitido. El solver tendrá que elegir días de inicio que eviten esa situación.

El ciclo interior busca si cada tarea ya empezó y todavía sigue trabajando ese día. En palabras simples: si empezó, y aún no han pasado todos sus días de duración, cuenta como activa.

## Variables para imprimir los resultados

El modelo guarda los posibles inicios como respuestas de sí/no: por ejemplo, “la tarea 1 empieza el día 0”, “empieza el día 1”, etc. Para imprimirlo más fácilmente, crea dos variables numéricas para cada tarea:

s  # día de inicio
e  # día de fin

La línea:

model.Add(s == sum(t * start_x[t, i] for t in range(horizon)))

convierte las respuestas de sí/no en el número del día. Si la tarea empieza el día 3, solo start_x[3, i] vale 1; la suma da 3.

Después:

model.Add(e == s + durations[i])

calcula el fin. Si empieza el día 3 y dura 2 días, termina el día 5.

## Resolver e imprimir

solver = cp_model.CpSolver()
status = solver.Solve(model)

Primero se crea el solver; luego se le pide encontrar valores que cumplan todas las reglas.

Si encuentra una solución, el código imprime el inicio y el fin de cada tarea. Si no encuentra ninguna, imprime que no hay una combinación posible.

Resumen: la restricción 3 evita que se usen más de 5 obreros en un día; el resto convierte las decisiones del solver en días de inicio y fin para mostrarlas.

# Pregunta 4

En un sistema de asignación, se tienen las variables PA (Proyecto A) y PB (Proyecto B), ambas con dominio de días {1,2,3,4,5}. Existen tres reglas condicionales estrictas:  
  
Si el Proyecto A se asigna al día 3, entonces el Proyecto B debe asignarse al día 4.  
Si el Proyecto A se asigna al día 4, entonces el Proyecto B debe asignarse al día 5.  
Si el Proyecto B se asigna al día 1, entonces el Proyecto A debe asignarse al día 5.  
Defina las variables booleanas reificadas que representen los "antecedentes" de estas condiciones (la parte del "Si...").  
(1 punto) Escriba las equivalencias formales para esas booleanas y cómo quedarían las implicaciones lógicas hacia las consecuencias.  
(2 puntos) Muestre la reificación bidireccional de los antecedentes y las implicaciones hacia los consecuentes.  
(2 puntos) Implemente estas tres reglas condicionales en OR-Tools utilizando variables booleanas explícitas y el método .OnlyEnforceIf().  

In [ ]:
from ortools.sat.python import cp_model

model = cp_model.CpModel()

PA = model.NewIntVar(1, 5, 'PA')
PB = model.NewIntVar(1, 5, 'PB')

# Variables booleanas explícitas para antecedentes
b1 = model.NewBoolVar('b1')
b2 = model.NewBoolVar('b2')
b3 = model.NewBoolVar('b3')
# Tenemos 3 condicionales y por eso creamos 3.

# Reificación Bidireccional: Vincula estrictamente el booleano al estado de la condición
model.Add(PA == 3).OnlyEnforceIf(b1) # si b1 es verdadero entonces PA es 3
model.Add(PA != 3).OnlyEnforceIf(b1.Not()) # si b1 es falso entonces PA no es 3

model.Add(PA == 4).OnlyEnforceIf(b2)
model.Add(PA != 4).OnlyEnforceIf(b2.Not())

model.Add(PB == 1).OnlyEnforceIf(b3)
model.Add(PB != 1).OnlyEnforceIf(b3.Not())

# Implicaciones condicionales a los consecuentes (Si antecedente es True -> Consecuencia)
model.Add(PB == 4).OnlyEnforceIf(b1) # Se completa la regla: Si PA es 3 entonces PB es 4ok
model.Add(PB == 5).OnlyEnforceIf(b2)
model.Add(PA == 5).OnlyEnforceIf(b3)

solver = cp_model.CpSolver()
status = solver.Solve(model)

if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    print("PA =", solver.Value(PA))
    print("PB =", solver.Value(PB))
    print("b1 =", solver.Value(b1))
    print("b2 =", solver.Value(b2))
    print("b3 =", solver.Value(b3))
else:
    print("No se encontró una solución.")

# de cuaderno

## pregunta 3

In [ ]:
from ortools.sat.python import cp_model

In [ ]:
class VarArraySolutionPrinter(cp_model.CpSolverSolutionCallback):
  def __init__(self, variables):
    cp_model.CpSolverSolutionCallback.__init__(self)
    self.__variables = variables
    self.__solution_count = 0

  def on_solution_callback(self):
    self.__solution_count += 1
    for v in self.__variables:
      print('%s=%i' % (v, self.Value(v)), end=' ')
    print()

  def solution_count(self):
    return self.__solution_count

In [ ]:
# Crear el modelo
model = cp_model.CpModel()

# horizonte  de 10 días
horizonte = 10

# --------------------------------------------------
# VARIABLES de tiempo (inicio, fin e intervalo para cada tarea)
# --------------------------------------------------

# tarea 1 duracion = 2
s1 = model.NewIntVar(0, horizonte, 'start_t1')
e1 = model.NewIntVar(0, horizonte, 'end_t1')
dur1 = 2
# s1 + dur1 == e1
int1 = model.NewIntervalVar(s1, dur1, e1, 'interval_t1')

# tarea 2 duracion = 3
s2 = model.NewIntVar(0, horizonte, 'start_t2')
e2 = model.NewIntVar(0, horizonte, 'end_t2')
dur2 = 3
int2 = model.NewIntervalVar(s2, dur2, e2, 'interval_t2')

# tarea 3 duracion = 1,
s3 = model.NewIntVar(0, horizonte, 'start_t3')
e3 = model.NewIntVar(0, horizonte, 'end_t3')
dur3 = 1
int3 = model.NewIntervalVar(s3, dur3, e3, 'interval_t3')

# --------------------------------------------------
# RESTRICCIÓN
# capacidad maxima de 5 obreros
# --------------------------------------------------
# Tarea 1  3 obreros
# Tarea 2  2 obreros
# Tarea 3  4 obreros

intervals = [int1, int2, int3]
demands = [3, 2, 4]
capacity = 5

model.AddCumulative(intervals, demands, capacity)



solver = cp_model.CpSolver()
printer = VarArraySolutionPrinter([s1, e1, s2, e2, s3, e3])
status = solver.SearchForAllSolutions(model, printer)

## pregunta 4

In [ ]:
# --------------------------------------------------
# MODELO
# --------------------------------------------------
model = cp_model.CpModel()
# --------------------------------------------------
# VARIABLES
# --------------------------------------------------

# Proyecto A
A = model.NewIntVar(1, 5, 'A')

# Proyecto B
B = model.NewIntVar(1, 5, 'B')

# --------------------------------------------------
# RESTRICCIÓN 1:
# Si A == 3 entonces B == 4
# a es la condiccion booleana  A == 3
# FORMAL
# a only enforce B = 4
# --------------------------------------------------

a = model.NewBoolVar('A_es_3')

model.Add(B == 4).OnlyEnforceIf(a)

# --------------------------------------------------
# RESTRICCIÓN 2:
# Si A == 4 entonces B == 5
# b es la condiccion booleana  A == 4
# FORMAL
# b only enforce B = 5
# --------------------------------------------------

b = model.NewBoolVar('A_es_4')

model.Add(B == 5).OnlyEnforceIf(b)

# --------------------------------------------------
# RESTRICCIÓN 3:
# Si B == 1 entonces A == 5
# c es la condiccion booleana  B == 1
# FORMAL
# c only enforce A = 5
# --------------------------------------------------

c = model.NewBoolVar('B_es_1')

model.Add(A == 5).OnlyEnforceIf(c)

solver = cp_model.CpSolver()
solution_printer = VarArraySolutionPrinter([A,B,a,b,c])
status = solver.SearchForAllSolutions(model, solution_printer)